Task 1

Download the dataset and load it

In [ ]:
!wget https://github.com/nlp-unibo/nlp-course-material/tree/main/2025-2026/Assignment%201/data

In [1]:
import json

# Load the files
with open("data/training.json", "r") as tr, \
     open("data/validation.json", "r") as te, \
     open("data/test.json", "r") as va:
    train_json = json.load(tr)
    val_json = json.load(te)
    test_json = json.load(va)

In [2]:
import pandas as pd
import numpy as np
from collections import Counter

# Create the dataframes (setting the index to id_EXIST)
train = pd.DataFrame.from_dict(train_json, orient="index").set_index("id_EXIST")
test = pd.DataFrame.from_dict(test_json, orient="index").set_index("id_EXIST")
val = pd.DataFrame.from_dict(val_json, orient="index").set_index("id_EXIST")

# Aggregate the labels (labels_task2)
mapping = {
    '-': 0,
    'DIRECT': 1,
    'JUDGEMENTAL': 2,
    'REPORTED': 3,
    'UNKNOWN': 4 # CAREFUL! 'UNKNOWN' doesn't appear in the assignment's
                 # specifications. Ask teacher for calrifications.
}

func = lambda row: \
    mapping[Counter(row.labels_task2).most_common(1)[0][0]]

train["labels"] = train.apply(
    func,
    axis = 1
)
test["labels"] = test.apply(
    func,
    axis = 1
)
val["labels"] = val.apply(
    func,
    axis = 1
)

# Drop spanish data
train = train[train.lang == "en"]
test = test[test.lang == "en"]
val = val[val.lang == "en"]

# Drop unnecessary columns
drop_cols = ["number_annotators", "annotators", "gender_annotators", 
    "age_annotators", "labels_task1", "labels_task2", "labels_task3", "split"]
train.drop(columns=drop_cols)
test.drop(columns=drop_cols)
val.drop(columns=drop_cols);


In [12]:
import re
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.tokenize import (word_tokenize,
                            sent_tokenize,
                            WhitespaceTokenizer);

nltk.download('omw-1.4');
nltk.download('wordnet');
nltk.download('averaged_perceptron_tagger');
nltk.download('averaged_perceptron_tagger_eng');
nltk.download('punkt_tab');
nltk.download('punkt');

tokenizer = WhitespaceTokenizer()
lemmatizer = WordNetLemmatizer()

# Clean the tweets and lemmatize
def get_wordnet_key(pos_tag):
    if pos_tag.startswith('J'):
        return wordnet.ADJ
    elif pos_tag.startswith('V'):
        return wordnet.VERB
    elif pos_tag.startswith('N'):
        return wordnet.NOUN
    elif pos_tag.startswith('R'):
        return wordnet.ADV
    else:          
        return 'n'

def lem_text(row):
    tokens = tokenizer.tokenize(row.tweet)
    tagged = pos_tag(tokens)
    words = [lemmatizer.lemmatize(word, get_wordnet_key(tag)) 
             for word, tag in tagged]
    return " ".join(words)

def cleaner(row):
    text = row.tweet
    text = text.lower()
    text = re.sub(r'https?:\/\/.\S+', '', text)
    text = re.sub(r'[@#].\S+', '', text)
    text = re.sub(
        "["
            u"\U0001F600-\U0001F64F"  # Emoticons
            u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # Transport & map symbols
            u"\U0001F1E0-\U0001F1FF"  # Flags
                                    "]+", '', text
    )
    text = re.sub(r'[^a-z^0-9^\s]*', '', text)
    text = ' '.join(text.split())
    return text

train["tweet"] = train.apply(
    cleaner,
    axis = 1
)
train["tweet"] = train.apply(
    lem_text,
    axis = 1
)
test["tweet"] = test.apply(
    cleaner,
    axis = 1
)
test["tweet"] = test.apply(
    lem_text,
    axis = 1
)
val["tweet"] = val.apply(
    cleaner,
    axis = 1
)
val["tweet"] = val.apply(
    lem_text,
    axis = 1
)

train['tweet'].head(
)

[nltk_data] Downloading package omw-1.4 to /home/ludi/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet to /home/ludi/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ludi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/ludi/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to /home/ludi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to /home/ludi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


id_EXIST
200001    ffs how about lay the blame on the bastard who...
200002    write a uni essay in my local pub with a coffe...
200003    it be 2021 not 1921 i dont appreciate that on ...
200004    this be unacceptable use her title a you do fo...
200005    make yourself a hard target basically boil dow...
Name: tweet, dtype: object